In [ ]:
! pip install duckdb

In [2]:
import duckdb

try:
    con.close() 
except NameError:
    pass

con = duckdb.connect("../dataset/goodreads.db") 
con.sql("SET memory_limit='4GB'")
con.sql("SET temp_directory='duckdb_tmp'")

### First round of filtration

In [3]:
con.sql("""
    CREATE OR REPLACE TABLE user_stats AS
    SELECT user_id,
           COUNT(*) AS n_interactions,
           SUM(CASE WHEN rating > 0 THEN 1 ELSE 0 END) AS n_rated
    FROM interactions
    GROUP BY user_id
""")

In [4]:
con.sql("""
    CREATE OR REPLACE TABLE book_stats AS
    SELECT book_id,
           COUNT(*) AS n_interactions,
           SUM(CASE WHEN rating > 0 THEN 1 ELSE 0 END) AS n_rated
    FROM interactions
    GROUP BY book_id
""")

In [5]:
con.sql("""
    SELECT 'users' AS what,
           COUNT(*) AS total,
           MEDIAN(n_rated) AS median_rated,
           quantile_cont(n_rated, 0.9) AS p90_rated,
           MAX(n_rated) AS max_rated,
           SUM(CASE WHEN n_rated >= 20 THEN 1 ELSE 0 END) AS with_20_plus_rated
    FROM user_stats
    UNION ALL
    SELECT 'books',
           COUNT(*),
           MEDIAN(n_rated),
           quantile_cont(n_rated, 0.9),
           MAX(n_rated),
           SUM(CASE WHEN n_rated >= 20 THEN 1 ELSE 0 END)
    FROM book_stats
""").show()

┌─────────┬─────────┬──────────────┬───────────┬───────────┬────────────────────┐
│  what   │  total  │ median_rated │ p90_rated │ max_rated │ with_20_plus_rated │
│ varchar │  int64  │    double    │  double   │  int128   │       int128       │
├─────────┼─────────┼──────────────┼───────────┼───────────┼────────────────────┤
│ users   │  876145 │         49.0 │     295.0 │     38884 │             639453 │
│ books   │ 2360643 │          4.0 │      42.0 │    281135 │             434531 │
└─────────┴─────────┴──────────────┴───────────┴───────────┴────────────────────┘



In [8]:
#use iterative k core filteration
con.sql("""
    CREATE OR REPLACE TABLE kcore AS
    SELECT user_id, book_id, rating, is_read, is_reviewed
    FROM interactions
    WHERE rating > 0
""")


for i in range(1, 11):
    before = con.sql("SELECT COUNT(*) FROM kcore").fetchone()[0]

    con.sql("""
        CREATE OR REPLACE TABLE kcore_next AS
        SELECT * FROM kcore
        WHERE user_id IN (SELECT user_id FROM kcore GROUP BY user_id HAVING COUNT(*) >= 20)
          AND book_id IN (SELECT book_id FROM kcore GROUP BY book_id HAVING COUNT(*) >= 20)
    """)
    con.sql("DROP TABLE kcore")
    con.sql("ALTER TABLE kcore_next RENAME TO kcore")

    after = con.sql("SELECT COUNT(*) FROM kcore").fetchone()[0]
    print(f"Round {i}: {before:,} -> {after:,} rows")
    if before == after:
        print("Stable, done.")
        break



con.sql("""
    SELECT COUNT(*) AS n_rows,
           COUNT(DISTINCT user_id) AS n_users,
           COUNT(DISTINCT book_id) AS n_books
    FROM kcore
""").show()

Round 1: 104,551,537 -> 94,313,824 rows
Round 2: 94,313,824 -> 94,045,805 rows
Round 3: 94,045,805 -> 94,001,311 rows
Round 4: 94,001,311 -> 93,997,057 rows
Round 5: 93,997,057 -> 93,996,052 rows
Round 6: 93,996,052 -> 93,995,938 rows
Round 7: 93,995,938 -> 93,995,919 rows
Round 8: 93,995,919 -> 93,995,919 rows
Stable, done.
┌──────────┬─────────┬─────────┐
│  n_rows  │ n_users │ n_books │
│  int64   │  int64  │  int64  │
├──────────┼─────────┼─────────┤
│ 93995919 │  626362 │  428655 │
└──────────┴─────────┴─────────┘



### Sampling

In [11]:
#sampling by users
#randomly select 10k users (pseudorandom) should give about 1.5m rows
con.sql("""
    CREATE OR REPLACE TABLE sample_users AS
    SELECT DISTINCT user_id
    FROM kcore
    ORDER BY hash(user_id)
    LIMIT 10000
""")

con.sql("""
    CREATE OR REPLACE TABLE sample_data AS
    SELECT * FROM kcore
    WHERE user_id IN (SELECT user_id FROM sample_users)
""")

con.sql("""
    SELECT COUNT(*) AS n_rows,
           COUNT(DISTINCT user_id) AS n_users,
           COUNT(DISTINCT book_id) AS n_books
    FROM sample_data
""").show()

┌─────────┬─────────┬─────────┐
│ n_rows  │ n_users │ n_books │
│  int64  │  int64  │  int64  │
├─────────┼─────────┼─────────┤
│ 1503365 │   10000 │  254864 │
└─────────┴─────────┴─────────┘



In [12]:
#filtration again
#first select a cutoff
con.sql("""
    WITH book_count AS (
        SELECT book_id, COUNT(*) AS n
        FROM sample_data
        GROUP BY book_id
    )
    SELECT t AS min_ratings_per_book,
           COUNT(*) FILTER (WHERE n >= t) AS books_kept,
           SUM(n) FILTER (WHERE n >= t) AS rows_kept
    FROM book_count, (VALUES (1), (3), (5), (10), (20)) AS cutoffs(t)
    GROUP BY t
    ORDER BY t
""").show()

┌──────────────────────┬────────────┬───────────┐
│ min_ratings_per_book │ books_kept │ rows_kept │
│        int32         │   int64    │  int128   │
├──────────────────────┼────────────┼───────────┤
│                    1 │     254864 │   1503365 │
│                    3 │      88360 │   1284324 │
│                    5 │      48927 │   1151902 │
│                   10 │      22644 │    983471 │
│                   20 │      10404 │    820376 │
└──────────────────────┴────────────┴───────────┘



In [13]:
#selected cutoff is 10
con.sql("""
    CREATE OR REPLACE TABLE sample_final AS
    SELECT * FROM sample_data
    WHERE book_id IN (
        SELECT book_id FROM sample_data
        GROUP BY book_id HAVING COUNT(*) >= 10
    )
""")

con.sql("""
    SELECT COUNT(*) AS n_rows,
           COUNT(DISTINCT user_id) AS n_users,
           COUNT(DISTINCT book_id) AS n_books
    FROM sample_final
""").show()

con.sql("""
    SELECT MIN(n) AS min_per_user, MEDIAN(n) AS median_per_user
    FROM (SELECT user_id, COUNT(*) AS n FROM sample_final GROUP BY user_id)
""").show()

┌────────┬─────────┬─────────┐
│ n_rows │ n_users │ n_books │
│ int64  │  int64  │  int64  │
├────────┼─────────┼─────────┤
│ 983471 │    9993 │   22644 │
└────────┴─────────┴─────────┘

┌──────────────┬─────────────────┐
│ min_per_user │ median_per_user │
│    int64     │     double      │
├──────────────┼─────────────────┤
│            1 │            55.0 │
└──────────────┴─────────────────┘



### SEcond round of filtration

In [14]:
#the previous filtration step left at least one user with a single rating, which we should remove...we need at least 20 ratings per person
#do another kcore filtration
con.sql("""
    CREATE OR REPLACE TABLE final_core AS
    SELECT * FROM sample_final
""")

for i in range(1, 11):
    before = con.sql("SELECT COUNT(*) FROM final_core").fetchone()[0]

    con.sql("""
        CREATE OR REPLACE TABLE final_next AS
        SELECT * FROM final_core
        WHERE user_id IN (SELECT user_id FROM final_core GROUP BY user_id HAVING COUNT(*) >= 20)
          AND book_id IN (SELECT book_id FROM final_core GROUP BY book_id HAVING COUNT(*) >= 10)
    """)
    con.sql("DROP TABLE final_core")
    con.sql("ALTER TABLE final_next RENAME TO final_core")

    after = con.sql("SELECT COUNT(*) FROM final_core").fetchone()[0]
    print(f"Round {i}: {before:,} -> {after:,} rows")
    if before == after:
        print("Stable, done.")
        break

con.sql("""
    SELECT COUNT(*) AS n_rows,
           COUNT(DISTINCT user_id) AS n_users,
           COUNT(DISTINCT book_id) AS n_books
    FROM final_core
""").show()

con.sql("""
    SELECT MIN(n) AS min_per_user, MEDIAN(n) AS median_per_user
    FROM (SELECT user_id, COUNT(*) AS n FROM final_core GROUP BY user_id)
""").show()

Round 1: 983,471 -> 968,813 rows
Round 2: 968,813 -> 964,346 rows
Round 3: 964,346 -> 963,561 rows
Round 4: 963,561 -> 963,189 rows
Round 5: 963,189 -> 963,020 rows
Round 6: 963,020 -> 962,922 rows
Round 7: 962,922 -> 962,922 rows
Stable, done.
┌────────┬─────────┬─────────┐
│ n_rows │ n_users │ n_books │
│ int64  │  int64  │  int64  │
├────────┼─────────┼─────────┤
│ 962922 │    8825 │   22070 │
└────────┴─────────┴─────────┘

┌──────────────┬─────────────────┐
│ min_per_user │ median_per_user │
│    int64     │     double      │
├──────────────┼─────────────────┤
│           20 │            65.0 │
└──────────────┴─────────────────┘



### Combine features into one table

In [4]:
# combine features into one table
#first load the genres
con.sql("""
    CREATE OR REPLACE TABLE genres_raw AS
    SELECT CAST(book_id AS BIGINT) AS book_id, genres
    FROM read_json_auto('../dataset_original/goodreads_book_genres_initial.json',
                        format='newline_delimited')
""")

con.sql("""
    SELECT
        (SELECT COUNT(*) FROM genres_raw) AS genre_rows,
        (SELECT COUNT(DISTINCT book_id) FROM genres_raw) AS distinct_genre_books,
        (SELECT COUNT(DISTINCT book_id) FROM final_core) AS books_in_final,
        (SELECT COUNT(DISTINCT f.book_id)
           FROM final_core f JOIN genres_raw g ON f.book_id = g.book_id) AS books_with_genres
""").show()

┌────────────┬──────────────────────┬────────────────┬───────────────────┐
│ genre_rows │ distinct_genre_books │ books_in_final │ books_with_genres │
│   int64    │        int64         │     int64      │       int64       │
├────────────┼──────────────────────┼────────────────┼───────────────────┤
│    2360655 │              2360655 │          22070 │             22070 │
└────────────┴──────────────────────┴────────────────┴───────────────────┘



In [5]:
# flatten genre data into one row per book one column per genre
#count how many books have no genre information
con.sql("""
    CREATE OR REPLACE TABLE genres_flat AS
    SELECT
        book_id,
        COALESCE(CAST(genres['history, historical fiction, biography'] AS BIGINT), 0) AS g_history,
        COALESCE(CAST(genres['fiction'] AS BIGINT), 0) AS g_fiction,
        COALESCE(CAST(genres['fantasy, paranormal'] AS BIGINT), 0)    AS g_fantasy,
        COALESCE(CAST(genres['mystery, thriller, crime'] AS BIGINT), 0) AS g_mystery,
        COALESCE(CAST(genres['poetry'] AS BIGINT), 0)  AS g_poetry,
        COALESCE(CAST(genres['romance'] AS BIGINT), 0) AS g_romance,
        COALESCE(CAST(genres['non-fiction'] AS BIGINT), 0) AS g_nonfiction,
        COALESCE(CAST(genres['children'] AS BIGINT), 0)    AS g_children,
        COALESCE(CAST(genres['young-adult'] AS BIGINT), 0) AS g_young_adult,
        COALESCE(CAST(genres['comics, graphic'] AS BIGINT), 0) AS g_comics
    FROM genres_raw
""")

con.sql("""
    SELECT
        COUNT(*) AS books_in_final,
        SUM(CASE WHEN g_history + g_fiction + g_fantasy + g_mystery + g_poetry
                    + g_romance + g_nonfiction + g_children + g_young_adult + g_comics = 0
                 THEN 1 ELSE 0 END) AS books_with_no_genre_counts
    FROM genres_flat
    WHERE book_id IN (SELECT DISTINCT book_id FROM final_core)
""").show()

┌────────────────┬────────────────────────────┐
│ books_in_final │ books_with_no_genre_counts │
│     int64      │           int128           │
├────────────────┼────────────────────────────┤
│          22070 │                          2 │
└────────────────┴────────────────────────────┘



In [7]:
# load authors file and check how many of the sampled books having a matching author 
con.sql("""
    CREATE OR REPLACE TABLE authors_raw AS
    SELECT
        CAST(author_id AS BIGINT) AS author_id,
        name,
        TRY_CAST(average_rating AS DOUBLE) AS author_avg_rating,
        TRY_CAST(ratings_count AS BIGINT) AS author_ratings_count,
        TRY_CAST(text_reviews_count AS BIGINT) AS author_text_reviews_count
    FROM read_json_auto('../dataset_original/goodreads_book_authors.json',
                        format='newline_delimited')
""")

con.sql("""
    SELECT
        (SELECT COUNT(*) FROM authors_raw) AS author_rows,
        (SELECT COUNT(DISTINCT author_id) FROM authors_raw) AS distinct_authors,
        (SELECT COUNT(DISTINCT b.book_id)
           FROM books_clean b
           WHERE b.book_id IN (SELECT book_id FROM final_core)) AS books_in_final,
        (SELECT COUNT(DISTINCT b.book_id)
           FROM books_clean b
           JOIN authors_raw a ON b.first_author_id = a.author_id
           WHERE b.book_id IN (SELECT book_id FROM final_core)) AS books_with_author
""").show()

┌─────────────┬──────────────────┬────────────────┬───────────────────┐
│ author_rows │ distinct_authors │ books_in_final │ books_with_author │
│    int64    │      int64       │     int64      │       int64       │
├─────────────┼──────────────────┼────────────────┼───────────────────┤
│      829529 │           829529 │          22070 │             22068 │
└─────────────┴──────────────────┴────────────────┴───────────────────┘



In [8]:
#finally, combine the tables: each row is a rating with the books features
#drop the 2 books with no genre and 2 books with no author here using inner joins
con.sql("""
    CREATE OR REPLACE TABLE combined AS
    SELECT
        f.user_id,
        f.book_id,
        f.rating,
        f.is_read,
        f.is_reviewed,
        b.title,
        b.description,
        b.language_code,
        b.publisher,
        b.format,
        b.is_ebook,
        b.avg_rating AS book_avg_rating,
        b.ratings_count AS book_ratings_count,
        b.text_reviews_count AS book_text_reviews_count,
        b.num_pages,
        b.publication_year,
        b.in_series,
        b.n_authors,
        a.name AS author_name,
        a.author_avg_rating,
        a.author_ratings_count,
        a.author_text_reviews_count,
        g.g_history, g.g_fiction, g.g_fantasy, g.g_mystery, g.g_poetry,
        g.g_romance, g.g_nonfiction, g.g_children, g.g_young_adult, g.g_comics
    FROM final_core f
    JOIN books_clean b ON f.book_id = b.book_id
    JOIN authors_raw a ON b.first_author_id = a.author_id
    JOIN genres_flat g ON f.book_id = g.book_id
    WHERE g.g_history + g.g_fiction + g.g_fantasy + g.g_mystery + g.g_poetry
        + g.g_romance + g.g_nonfiction + g.g_children + g.g_young_adult + g.g_comics > 0
""")

con.sql("""
    SELECT COUNT(*) AS n_rows,
           COUNT(DISTINCT user_id) AS n_users,
           COUNT(DISTINCT book_id) AS n_books
    FROM combined
""").show()

┌────────┬─────────┬─────────┐
│ n_rows │ n_users │ n_books │
│ int64  │  int64  │  int64  │
├────────┼─────────┼─────────┤
│ 962899 │    8825 │   22068 │
└────────┴─────────┴─────────┘



### Post cleaning check (check for missing values)

In [ ]:
# PER ROW/RATING
con.sql("""
    SELECT
        COUNT(*) AS n_rows,
        SUM(CASE WHEN description IS NULL THEN 1 ELSE 0 END) AS description_null,
        SUM(CASE WHEN publisher IS NULL THEN 1 ELSE 0 END) AS publisher_null,
        SUM(CASE WHEN format IS NULL THEN 1 ELSE 0 END) AS format_null,
        SUM(CASE WHEN num_pages IS NULL THEN 1 ELSE 0 END) AS pages_null,
        SUM(CASE WHEN publication_year IS NULL THEN 1 ELSE 0 END) AS year_null,
        SUM(CASE WHEN language_code = 'unknown' THEN 1 ELSE 0 END) AS language_unknown,
        SUM(CASE WHEN book_avg_rating IS NULL THEN 1 ELSE 0 END) AS book_avg_null,
        SUM(CASE WHEN author_avg_rating IS NULL THEN 1 ELSE 0 END) AS author_avg_null
    FROM combined
""").show()

┌────────┬──────────────────┬────────────────┬─────────────┬────────────┬───────────┬──────────────────┬───────────────┬─────────────────┐
│ n_rows │ description_null │ publisher_null │ format_null │ pages_null │ year_null │ language_unknown │ book_avg_null │ author_avg_null │
│ int64  │      int128      │     int128     │   int128    │   int128   │  int128   │      int128      │    int128     │     int128      │
├────────┼──────────────────┼────────────────┼─────────────┼────────────┼───────────┼──────────────────┼───────────────┼─────────────────┤
│ 962899 │            13494 │         152993 │      150891 │      39467 │    140590 │            53990 │             0 │               0 │
└────────┴──────────────────┴────────────────┴─────────────┴────────────┴───────────┴──────────────────┴───────────────┴─────────────────┘



In [10]:
#PER BOOK
con.sql("""
    WITH b AS (
        SELECT DISTINCT book_id, description, publisher, format,
               num_pages, publication_year, language_code
        FROM combined
    )
    SELECT
        COUNT(*) AS n_books,
        ROUND(100.0 * SUM(CASE WHEN description IS NULL THEN 1 ELSE 0 END) / COUNT(*), 1) AS description_pct,
        ROUND(100.0 * SUM(CASE WHEN publisher IS NULL THEN 1 ELSE 0 END) / COUNT(*), 1) AS publisher_pct,
        ROUND(100.0 * SUM(CASE WHEN format IS NULL THEN 1 ELSE 0 END) / COUNT(*), 1) AS format_pct,
        ROUND(100.0 * SUM(CASE WHEN num_pages IS NULL THEN 1 ELSE 0 END) / COUNT(*), 1) AS pages_pct,
        ROUND(100.0 * SUM(CASE WHEN publication_year IS NULL THEN 1 ELSE 0 END) / COUNT(*), 1) AS year_pct,
        ROUND(100.0 * SUM(CASE WHEN language_code = 'unknown' THEN 1 ELSE 0 END) / COUNT(*), 1) AS language_unknown_pct
    FROM b
""").show()

┌─────────┬─────────────────┬───────────────┬────────────┬───────────┬──────────┬──────────────────────┐
│ n_books │ description_pct │ publisher_pct │ format_pct │ pages_pct │ year_pct │ language_unknown_pct │
│  int64  │     double      │    double     │   double   │  double   │  double  │        double        │
├─────────┼─────────────────┼───────────────┼────────────┼───────────┼──────────┼──────────────────────┤
│   22068 │             2.8 │          18.9 │       18.3 │       7.5 │     15.8 │                 10.8 │
└─────────┴─────────────────┴───────────────┴────────────┴───────────┴──────────┴──────────────────────┘



In [11]:
con.close()